In [1]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
import numpy as np
from model.dataloader import load_shards, get_svm_data

In [2]:
# Function to calculate mean and std of training and testing sets
def calculate_stats(tf_dataset, num_samples=5000):
    images_iterator = tf_dataset.unbatch().take(num_samples).as_numpy_iterator()
    all_images = np.stack([img for img, _ in images_iterator])

    means = np.mean(all_images, axis=(0, 1, 2))
    stds = np.std(all_images, axis=(0, 1, 2))

    return means.tolist(), stds.tolist()

In [3]:
# Load the data from the drive
# Use your local filepath here
file_pattern = r'G:/.shortcut-targets-by-id/1abX3CWvYUSJM3cGg6r_GeHAVeNoYH6Ul/CS6140_Project_Data/koppen_shard_part_*.tfrecord.gz'
all_files = tf.io.gfile.glob(file_pattern)
print(len(all_files), 'shards loaded')

# Split training and testing data
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

1500 shards loaded


In [4]:
loader_batch_size = 32

# First pass to calculate normalization statistics WITHOUT flattening to SVM features
raw_for_stats = load_shards(train_files, batch_size=loader_batch_size, stats=None)
train_means, train_stds = calculate_stats(raw_for_stats)

# Second pass to generate normalized training/testing data WITH flattening to SVM features
train_raw = load_shards(
    train_files,
    batch_size=loader_batch_size,
    stats=(train_means, train_stds),
    is_svm=True
)
test_raw = load_shards(
    test_files,
    batch_size=loader_batch_size,
    stats=(train_means, train_stds),
    is_svm=True
)

# Convert TF datasets into numpy arrays
X_train, y_train = get_svm_data(train_raw)
X_test, y_test = get_svm_data(test_raw)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(47467, 60)
(47467,)
(11735, 60)
(11735,)
